**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/asari-2009"], check=True)
    os.chdir("/content/encore/entries/asari-2009")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Asari & Zador (2009) from CRCNS ac-1

**Paper.** Asari H, Zador AM (2009). Long-lasting context dependence constrains neural encoding
models in rodent auditory cortex. *J Neurophysiol* 102:2638-2656.
[doi:10.1152/jn.00577.2009](https://doi.org/10.1152/jn.00577.2009). The claims file
(`claims/asari-2009.yaml`) was written from the version of record; the online supplement was not
available to its readers.

**Data set.** CRCNS ac-1 (Asari, Wehr, Machens & Zador 2009),
[doi:10.6080/K0KW5CXR](https://doi.org/10.6080/K0KW5CXR), read from the public bucket
`crcnsarchive` (`ac-1/`). CRCNS data sets are not versioned: the version is recorded as
`unversioned`, and the MD5 sums the data set publishes in `checksums.md5` for the three zip files
read here are printed below, so that a later change can be detected. There is no other version to
choose; the paper cites the data set by no version.

**What the data set holds.** Asari's recordings: whole-cell current-clamp recordings in A1 and
cell-attached recordings in MGB, one MATLAB file per recording, with the analysis code the
paper used (`asari/Tools/CxtDep.m`). Wehr's recordings of Machens et al. (2004) are in the same
data set. No voltage-clamp recording is in it (checked below).

**Method details taken from the paper's code.** The paper's own analysis script, `CxtDep.m`, is in
the data set. Where the Methods leave a step open, this notebook follows that script and says so;
`asari.py` lists each such step.

| Claim | Tested here |
|---|---|
| methods-recording-counts | tested (voltage-clamp counts not in the data set) |
| a1-mgb-firing-rates | tested |
| fig2-example-reliability | tested |
| fig2-example-long-context | tested |
| fig3a-proportion-significant | tested |
| fig3a-significance-decay | tested |
| fig3b-fractional-power-decay | tested |
| timescale-longer-than-membrane | tested |
| fig4-predictability-upper-bound | tested |
| fig4-strf-performance-low | tested in `strf_wehr.ipynb` (Wehr's recordings) |
| fig5-example-conductances | blocked: no voltage-clamp data |
| fig6-conductance-decay | blocked: no voltage-clamp data |
| fig7-example-frequency-context | searched; blocked, no recording with that design (a weaker check is shown) |
| fig8a-proportion-by-property | tested (property sets rebuilt from the metadata) |
| fig8a-significance-fits | tested |
| fig8b-decay-by-property | tested |
| fig8c-relative-effects | tested |
| fig8-natural-synthetic-probes-alike | tested |
| fig9-mgb-fast-decay | tested |
| fig9-example-first-bin | tested |
| figs1-silence-probe | tested |

`ASARI_SUBSET`, when set, limits the pass over the files to that many files per group; it is only
for development, and the record rests on a run without it.

In [ ]:
import os
import time
import zlib
import resource
import warnings
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import asari

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
SUBSET = int(os.environ.get("ASARI_SUBSET", "0"))
SF = asari.SF
started = time.time()

## The files

In [ ]:
md5 = asari.checksums()
print("MD5 sums published in ac-1/checksums.md5 (data set doi:%s, version unversioned):" % asari.DOI)
for k, v in md5.items():
    print(f"  {v}  {k}")
listing = {}
for zname in (asari.MAIN_ZIP,) + asari.RESULT_ZIPS:
    infos = asari.open_zip(zname).infolist()
    listing[zname] = infos
    mats = [i for i in infos if i.filename.endswith(".mat")]
    print(f"{zname}: {len(infos)} members, {len(mats)} .mat files, {sum(i.file_size for i in infos) / 1e9:.2f} GB,"
          f" .mat files stored without compression: {all(i.compress_type == 0 for i in mats)}")
tops = pd.Series([i.filename.split('/')[0] + '/' + (i.filename.split('/')[1] if '/' in i.filename else '') for i in listing[asari.MAIN_ZIP]])
print("\nTop folders of crcns-ac1.zip:")
print(tops.str.split('/').str[0].value_counts().to_string())

In [ ]:
rows = pd.DataFrame(asari.summary())
idx = asari.member_index()
rows["in_zip"] = rows.file.isin(idx.keys())
print("asari/Summary.mat: experiment groups")
print(rows.groupby(["group", "group_description", "reference"]).agg(cells=("cell", "nunique"), files=("file", "size")).to_string())
print("\nFiles named in Summary.mat but not in the result zips:", rows.loc[~rows.in_zip, "file"].tolist())
print("Result-zip files not named in Summary.mat:", sorted(set(idx) - set(rows.file)))
dup = rows[rows.duplicated("file", keep=False)]
print("Files listed under more than one cell or group:", dup.file.unique().tolist())

## One pass over the recordings

Every recording is streamed once. For the A1 recordings of the natural sound group (Summary.mat
group 0) the pass forms the probe responses as `CxtDep.m` does and computes, per probe, the
significance measure with its 1000-shuffle noise floor, the fractional powers, the trial-to-trial
reliability and the autocorrelation of the trial-to-trial noise. For the synthetic sound group
(group 2) it computes the same two measures per probe and sound property, with only the
conditioning stimuli that differ in that property as contexts (`asari.property_families`), once for every set a probe was tested with,
each set's contexts cut to the smallest number of trials among them. For the MGB recordings (group 1)
it detects spikes and forms the PSTHs per context. For every recording it counts spikes in four windows:
the stimulus periods and the silences between them, the 1 s before each stimulus, and the probe periods. Each file's shuffles use a generator seeded from the file name.

In [ ]:
todo = rows[rows.in_zip].drop_duplicates("file")
if SUBSET:
    todo = todo.groupby("group").head(SUBSET)

def process(row):
    m = asari.load_recording(row.file)
    kind = str(m["param"].recording.type)
    context = row.description != "tuning curve"
    if kind == "Cell-attached":
        out = asari.mgb_file_pass(m, do_context=(row.group == 1 and context))
    else:
        rng = np.random.default_rng(zlib.crc32(row.file.encode()))
        boot_rng = np.random.default_rng([zlib.crc32(row.file.encode()), 1])   # trial resampling of Figure 8C, apart from rng
        out = asari.a1_file_pass(m, do_context=(row.group == 0 and context), rng=rng, boot_rng=boot_rng)
        info = out["info"]
        if row.group == 2 and context and (info["rep"] or 0) >= 4:
            out["synthetic"] = asari.synthetic_file_pass(m, rng=rng, boot_rng=boot_rng)
        if row.group == 2 and context:
            trig = asari.sequence_triggers(m["triggers"])
            out["stimuli"] = asari.stimulus_table(m["param"])
            out["families"] = asari.property_families(m["param"], trig[0]["type"] if trig else "")
            out["notes_lines"] = [l.strip() for l in asari.notes_text(m["param"]).splitlines()
                                  if l.strip() and not l.lstrip().startswith(("Seq.", "Stimulus sequences"))]
    del m
    out.update(file=row.file, cell=row.cell, group=row.group, file_description=row.description)
    return out

t0 = time.time()
with ThreadPoolExecutor(3) as ex:
    results = list(ex.map(process, todo.itertuples()))
print(f"{len(results)} files streamed and reduced in {(time.time() - t0) / 60:.1f} min; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

In [ ]:
files = pd.DataFrame([{**r["info"], "file": r["file"], "cell": r["cell"], "group": r["group"],
                       "probes_with_silence": r["info"]["n_probes"]["with_silence"],
                       "probes_without_silence": r["info"]["n_probes"]["without_silence"]} for r in results])
files["tuning"] = files.description.str.startswith("tuning")
A1 = []
for r in results:
    for p in r.get("probes", []) if r["group"] == 0 else []:
        A1.append({**p, "cell": r["cell"], "file": r["file"]})
nat = [p for p in A1 if p["probe"] != asari.BLANK]
sil = [p for p in A1 if p["probe"] == asari.BLANK]
MGB = [{**p, "cell": r["cell"], "file": r["file"]} for r in results if r["group"] == 1 for p in r.get("probes", [])]
print(len(A1), "A1 probes formed (", len(nat), "natural sound fragments,", len(sil), "silence probes );", len(MGB), "MGB probes")

## Data check

In [ ]:
print("Recordings by Summary.mat group, recording type, unit and sampling rate:")
print(files.groupby(["group", "recording_type", "unit", "sf"]).agg(cells=("cell", "nunique"), files=("file", "size")).to_string())
print("\nNo file has a current unit (pA or nA); there are no voltage-clamp recordings:",
      not files.unit.str.contains("A").any())
print("Every file is sampled at 10 kHz, as the Methods say:", bool((files.sf == 10000).all()))
dups = files[files.duplicated("trace_md5", keep=False)].sort_values("trace_md5")
print("Recordings whose traces are identical (MD5 of the stored samples):", dups[["file", "cell", "group"]].values.tolist() if len(dups) else "none")
ctx = files[~files.tuning]
print("\nSequence (context) recordings, by group and stimulus type:")
print(ctx.assign(types=ctx.trigger_types.map(lambda t: "+".join(t))).groupby(["group", "types"]).agg(
    cells=("cell", "nunique"), files=("file", "size"), rep_min=("rep", "min"), rep_max=("rep", "max"),
    seq_min=("n_seq", "min"), seq_max=("n_seq", "max")).to_string())
print("\nAnimals (date and animal code) per group:", files.groupby("group").animal.nunique().to_dict())
print("Ages (days) in the files, A1 and MGB:", sorted(files.age.unique(), key=lambda a: (len(a), a)))
vm = np.array([v for v in files.loc[files.recording_type == "Whole-cell", "vm_percentiles_mv"]])
print("\nWhole-cell membrane potential, mV, over files (1st, 50th, 99th percentile of each file): "
      f"medians range {vm[:, 1].min():.1f} to {vm[:, 1].max():.1f}; 1st percentile min {vm[:, 0].min():.1f}; 99th max {vm[:, 2].max():.1f}")
lens = pd.Series([p["T"] / SF for p in nat])
print(f"\nNatural sound probe durations: {lens.min():.2f} to {lens.max():.2f} s (median {lens.median():.2f}); "
      f"contexts per probe {min(len(p['contexts']) for p in nat)} to {max(len(p['contexts']) for p in nat)}; "
      f"trials per context {min(p['rep'] for p in nat)} to {max(p['rep'] for p in nat)}")
print("Silence probes are", sorted({p['T'] / SF for p in sil}), "s long (CxtDep.m: 2.5 s)")

The stimulus triggers and the traces are in the same file and share its sample clock. As a check
that the trigger times mark the sound onsets, the cell below compares the change of the membrane
potential (A1) or of the firing rate (MGB) across each sequence onset, 200 ms after against 200 ms
before, with the same contrast at times 3 s earlier, inside the preceding silence.

In [ ]:
def onset_table(group, label):
    out = []
    for r in results:
        if r["group"] != group or "onset" not in r or r["file_description"] == "tuning curve":
            continue
        a, b = r["onset"]["trigger"], r["onset"]["shifted"]
        if len(a) > 2:
            out.append({"file": r["file"].split("/")[-1], "trigger": a.mean(), "shifted": b.mean(),
                        "p_trigger_vs_shifted": stats.wilcoxon(a[:len(b)], b[:len(a)]).pvalue if len(b) > 2 else np.nan})
    out = pd.DataFrame(out)
    print(f"{label}: {len(out)} files; mean change at the trigger {out.trigger.mean():.2f}, at the shifted times {out.shifted.mean():.2f}; "
          f"files with a larger absolute change at the trigger: {(out.trigger.abs() > out.shifted.abs()).sum()}; "
          f"files with p < 0.05 (Wilcoxon over onsets): {(out.p_trigger_vs_shifted < 0.05).sum()}")
    return out

on_a1 = onset_table(0, "A1 natural sound recordings, mV")
on_mgb = onset_table(1, "MGB recordings, spikes/s")

## methods-recording-counts

> Of 194 A1 cells recorded in current clamp, 123 met the analysis criterion (39 tested with natural
> sounds, 14 with TORCs, 6 with DMRs, 39 with modulated harmonic tones, 27 with modulated colored
> noise); 14 of 42 voltage-clamped cells and 14 of 16 MGB cells were analysed.

The rule (Methods, Context dependence at subthreshold voltage level; Table 1 legend): a probe
counts when it was tested with at least two conditioning stimuli, at least 4 trials each in A1
current clamp and at least 10 in MGB. Trials per context are the file's "Effective repeats". Both
readings of silence (counted as a context, or not) are run. The cells behind the difference between
133 and 123 are listed by ensemble, with the properties each was tested with in Figure 8.

In [ ]:
def counts(sel, min_rep):
    out = {}
    for reading in ("with_silence", "without_silence"):
        col = "probes_" + reading
        ok = sel[(sel.rep.fillna(0) >= min_rep) & (sel[col] > 0)]
        out[reading] = {"cells": ok.cell.nunique(), "probes": int(ok[col].sum())}
    return out

a1_nat = ctx[(ctx.group == 0)]
mgb = ctx[(ctx.group == 1)]
a1_all = ctx[(ctx.recording_type == "Whole-cell")]
c_nat, c_mgb, c_all = counts(a1_nat, 4), counts(mgb, 10), counts(a1_all, 4)
print("A1 natural sound group (Summary.mat group 0):", c_nat)
print("MGB (group 1):", c_mgb)
print("All A1 whole-cell recordings with sequences:", c_all)
print("\nA1 whole-cell cells in the data set, any recording:", files[files.recording_type == "Whole-cell"].cell.nunique(),
      "; MGB cells:", files[files.recording_type == "Cell-attached"].cell.nunique())
print("Probes formed in this notebook from the natural sound group, silence probes excluded:", len(nat),
      "in", len({p['cell'] for p in nat}), "cells")
def ens(t):
    t = set(t)
    for k, v in (("torc", "TORC"), ("dmr", "DMR"), ("synthetic", "MHT"), ("pinknoise", "MCN"), ("naturalistic", "naturalistic (SoundTrans variants)")):
        if k in t:
            return v
    return "+".join(sorted(t))
syn = ctx[(ctx.group == 2)].copy()
syn["ensemble"] = syn.trigger_types.map(ens)
ok = syn[(syn.rep.fillna(0) >= 4) & (syn.probes_with_silence > 0)]
print("\nSynthetic group (group 2), cells meeting the rule, by ensemble (paper: TORC 14, DMR 6, MHT 39, MCN 27):")
print(ok.groupby("ensemble").agg(cells=("cell", "nunique"), files=("file", "size"), probes=("probes_with_silence", "sum")).to_string())
print("Cells of group 2 meeting the rule:", ok.cell.nunique(), "; group 0 and group 2 together:",
      pd.concat([a1_nat[(a1_nat.rep >= 4) & (a1_nat.probes_with_silence > 0)], ok]).cell.nunique(), "(paper: 123)")
print("Cells in both group 0 and group 2:", sorted(set(a1_nat.cell) & set(syn.cell)))

tested = {}
for r in results:
    for t in r.get("synthetic", []):
        tested.setdefault(r["cell"], set()).add(t["property"])
cell_ens = ok.groupby("cell").ensemble.agg(lambda v: "+".join(sorted(set(v))))
tab13 = pd.DataFrame({"ensembles": cell_ens, "properties_tested": [", ".join(p for p in asari.PROPERTIES if p in tested.get(c, ())) for c in cell_ens.index]})
tab13["any_property_test"] = tab13.properties_tested != ""
print("\nGroup 2 cells meeting the rule, by ensemble: all, and those with at least one probe tested with a set of stimuli that differ in one property (Figure 8):")
print(tab13.groupby("ensembles").any_property_test.agg(cells="size", with_property_test="sum").to_string())
print("Group 2 cells meeting the rule with a property test:", int(tab13.any_property_test.sum()), "; without:", int((~tab13.any_property_test).sum()))
print("\nThe 96 group 2 cells meeting the rule (with the 39 natural sound cells, 2 of them shared, these make the 133):")
print(tab13.to_string())


**Result: partially reproduced.** The natural sound group gives exactly 305 probes in 39 cells and the MGB group 93 probes in 14 cells under the paper's rule, with silence counted as a context or not. All A1 recordings with sequences give 133 cells meeting the rule against the paper's 123: the 39 natural sound cells and 96 synthetic sound cells, 2 of them in both. The synthetic cells split by ensemble as TORC 14 and MCN 27 (as in the paper), DMR 15 against 6, MHT 26 against 39, and 14 cells with SoundTrans variants of natural sounds, which the paper's list does not name; every one of the 96 has at least one probe tested with a one-property set. The listing does not show which 123 cells the paper counted. The voltage-clamp counts (29 probes, 14 cells) cannot be tested: the data set holds no voltage-clamp recording.

## a1-mgb-firing-rates

> Firing rates were low in A1 (spontaneous 0.47 +/- 0.61 Hz, evoked 0.57 +/- 0.77 Hz, mean +/- SD,
> 194 cells) and high in MGB (spontaneous 0.78 +/- 1.25 Hz, evoked 11.4 +/- 16.9 Hz, 16 cells).

The criterion names two readings of each window. Spontaneous: the silences between stimuli (split
as `GetEvoked.m` splits a recording) or a pre-stimulus baseline, here the 1 s before each stimulus
(cut to the silence before it; a standard choice). Evoked: every stimulus period, or the probe
periods analysed (the fragments of the first 'Effective repeats' presentations of each sequence).
A1 spikes are detected two ways (`asari.a1_spikes`): the starting values of the code's manual
detector, and a 10 mV/ms upstroke. MGB spikes use the code's automatic detector (threshold two
thirds of the largest filtered value) and, as a second reading, the same detector with half that
threshold. Each cell's rate pools its files; the readings are all of a cell's files, or only its
sequence (context) files.

In [ ]:
def cell_rates(kind, method, only_context, sw, ew):
    acc = {}
    for r in results:
        if r["info"]["recording_type"] != kind or method not in r["rates"]:
            continue
        if only_context and r["file_description"] == "tuning curve":
            continue
        rr = r["rates"][method]
        a = acc.setdefault(r["cell"], [0, 0.0, 0, 0.0])
        a[0] += rr[ew][0]; a[1] += rr[ew][1]; a[2] += rr[sw][0]; a[3] += rr[sw][1]
    return pd.DataFrame([{"cell": c, "evoked": a[0] / a[1] if a[1] else np.nan, "spont": a[2] / a[3] if a[3] else np.nan}
                         for c, a in acc.items()])

rate_rows = []
for kind, methods in (("Whole-cell", ("code", "dvdt")), ("Cell-attached", ("code", "half"))):
    for method in methods:
        for only in (False, True):
            for sw in ("silence", "baseline"):
                for ew in ("stim", "probe"):
                    df = cell_rates(kind, method, only, sw, ew)
                    rate_rows.append({"area": "A1" if kind == "Whole-cell" else "MGB", "detector": method,
                                      "files": "context only" if only else "all", "spont_window": sw, "evoked_window": ew,
                                      "cells_spont": int(df.spont.notna().sum()), "cells_evoked": int(df.evoked.notna().sum()),
                                      "spont_mean": df.spont.mean(), "spont_sd": df.spont.std(), "evoked_mean": df.evoked.mean(),
                                      "evoked_sd": df.evoked.std(), "evoked_median": df.evoked.median()})
rates = pd.DataFrame(rate_rows)
exp_r = {"A1": (0.47, 0.57), "MGB": (0.78, 11.4)}
rates["ok_spont"] = [abs(m - exp_r[a][0]) <= 0.1 * exp_r[a][0] for a, m in zip(rates.area, rates.spont_mean)]
rates["ok_evoked"] = [abs(m - exp_r[a][1]) <= 0.1 * exp_r[a][1] for a, m in zip(rates.area, rates.evoked_mean)]
print(rates.round(3).to_string(index=False))
print("\nPaper: A1 spont 0.47 (SD 0.61), evoked 0.57 (SD 0.77), 194 cells; MGB spont 0.78 (SD 1.25), evoked 11.4 (SD 16.9), 16 cells.")
print("Spontaneous windows: 'silence' the silences between stimuli, 'baseline' the 1 s before each stimulus; evoked windows: 'stim' every stimulus period, 'probe' the probe periods analysed.")
key = ["files", "spont_window", "evoked_window"]
a1r, mgr = rates[rates.area == "A1"], rates[rates.area == "MGB"]
both = a1r.merge(mgr, on=key, suffixes=("_a1", "_mgb"))
both["all_four"] = both.ok_spont_a1 & both.ok_evoked_a1 & both.ok_spont_mgb & both.ok_evoked_mgb
both["partial_rule"] = (both.spont_mean_a1 < 1) & (both.evoked_mean_a1 < 1) & (both.evoked_mean_mgb > 5)
print(f"\nReadings (A1 detector x MGB detector x files x windows): {len(both)}; with all four means within tolerance: {int(both.all_four.sum())}")
print("A1 means below 1 Hz in every reading:", bool(((a1r.spont_mean < 1) & (a1r.evoked_mean < 1)).all()))
print(f"MGB evoked mean above 5 Hz: {int((mgr.evoked_mean > 5).sum())} of {len(mgr)} MGB readings; largest {mgr.evoked_mean.max():.2f} Hz; readings meeting the partial rule: {int(both.partial_rule.sum())} of {len(both)}")
print("MGB readings with the evoked mean above 5 Hz:")
print(mgr[mgr.evoked_mean > 5][["detector", "files", "spont_window", "evoked_window", "spont_mean", "evoked_mean"]].round(3).to_string(index=False))

**Result: partially reproduced.** No reading of windows, files and detectors gives all four means within tolerance (0 of 32). A1 rates are below 1 Hz under every reading; with the 10 mV/ms detector, over all files with the silences as spontaneous periods, the A1 spontaneous mean is 0.474 Hz (SD 0.603; paper 0.47, SD 0.61) but the evoked mean is 0.695 Hz (0.722 over the probe periods) against 0.57. The MGB evoked mean is 4.99 to 5.07 Hz with the code's detector and 7.07 to 7.15 Hz with half its threshold, against 11.4 Hz; the MGB spontaneous mean is within tolerance with the 1-s baseline and the code's detector (0.72 to 0.82 Hz) or with the silences and the lower threshold (0.77 to 0.82 Hz). A1 means below 1 Hz with the MGB evoked mean above 5 Hz, the criterion's rule for partial, holds in 28 of the 32 readings. The data set holds 145 A1 and 14 MGB cells, not 194 and 16.

## fig2-example-reliability

> The example A1 neuron of Figure 2 had high trial-to-trial reliability (0.74 +/- 0.08 for the
> shown probe and 0.61 +/- 0.07 over the seven natural sound fragments tested), and the mean
> correlation over the population was 0.31 +/- 0.09.

Per probe, the correlation between trials of one context over the probe period (median-filtered
membrane potential), averaged over the contexts. Readings: the mean over all pairs of trials, or
each trial against the mean of the others; the population over probes or over cells.

In [ ]:
rel = pd.DataFrame([{"cell": p["cell"], "file": p["file"].split("/")[-1], "probe": p["probe"], "pairs": p["rel_pairs"],
                     "loo": p["rel_loo"]} for p in nat])
for reading in ("pairs", "loo"):
    per_cell = rel.groupby("cell")[reading].mean()
    print(f"{reading:5s}: over {len(rel)} probes {rel[reading].mean():.3f} +/- {rel[reading].std():.3f};"
          f" over {len(per_cell)} cells {per_cell.mean():.3f} +/- {per_cell.std():.3f}"
          f"  (paper 0.31 +/- 0.09; within 0.22 to 0.40: probes {abs(rel[reading].mean() - 0.31) <= 0.09}, cells {abs(per_cell.mean() - 0.31) <= 0.09})")
print("\nPer cell (pairs reading):", rel.groupby('cell').pairs.mean().describe().round(3).to_dict())
# Candidate example cells: seven natural sound fragments, six effective repeats
cand = files[(files.group == 0) & (files.n_stimuli == 7) & (files.rep == 6)].file
ex = rel[rel.file.isin([f.split("/")[-1] for f in cand])]
print("\nFiles with seven natural sound fragments and six effective repeats:", len(cand))
print(ex.groupby("file").agg(n=("probe", "size"), pairs_mean=("pairs", "mean"), pairs_sd=("pairs", "std"), pairs_max=("pairs", "max"),
                            loo_mean=("loo", "mean"), loo_sd=("loo", "std"), loo_max=("loo", "max")).round(3).to_string())

**Result: reproduced.** Over the 39 cells the mean correlation between trials is 0.310 +/- 0.092 (pairs of trials, averaged per cell; paper 0.31 +/- 0.09); over the 305 probes it is 0.319 +/- 0.129. One of the five recordings with seven natural sound fragments and six repeats, 20051222-ha1-001-002, gives 0.611 +/- 0.081 over its seven probes and 0.737 for its best probe (paper 0.61 +/- 0.07 and 0.74). The leave-one-out reading gives higher values (0.437 over cells).

## fig2-example-long-context

> In an example A1 neuron, changing the conditioning stimulus changed the response to the probe
> for about 4 s; responses did not differ between 2 and 4 s after probe onset and diverged again
> after about 4 s.

Candidates: probes of cells with seven natural sound fragments whose contexts are silence and two
fragments, with six trials in at least one context. For each, every significant period.

In [ ]:
rows_ex = []
for p in nat:
    f = files.set_index("file").loc[p["file"]]
    if f.n_stimuli != 7 or len(p["contexts"]) != 3 or asari.BLANK not in p["contexts"]:
        continue
    trials = {c: n * f.rep for c, n in p["n_occurrences"].items()}
    if 6 not in trials.values():
        continue
    rr = asari.runs(p["signif"])
    early = any(a < 2 for a, b in rr)
    late = any(b > 4 for a, b in rr)
    mid = any(a < 4 and b > 2 for a, b in rr)
    rows_ex.append({"file": p["file"].split("/")[-1], "probe": p["probe"], "contexts": p["contexts"], "trials": trials,
                    "T_s": p["T"] / SF, "runs": [(round(a, 2), round(b, 2)) for a, b in rr], "n_runs": len(rr),
                    "before_2s": early, "between_2_4s": mid, "after_4s": late, "pattern": early and late and not mid})
ex2 = pd.DataFrame(rows_ex)
print(len(ex2), "candidate probes")
print(ex2.drop(columns="runs").to_string(index=False) if len(ex2) else "none")
print("\nEvery significant period (s) of each candidate with one:")
for _, r in (ex2[ex2.n_runs > 0] if len(ex2) else ex2).iterrows():
    print(f"  {r.file} probe {r.probe}: " + ", ".join(f"{a:.2f}-{b:.2f}" for a, b in r.runs))
late4 = sum(any(b > 4 for a, b in asari.runs(p["signif"])) for p in nat)
print(f"\nNatural sound probes with a significant period later than 4 s: {late4} of {len(nat)}")
ex_cell = rel[rel.file == "20051222-ha1-001-002.mat"]
print("\nReliability (pairs of trials) of each probe of 20051222-ha1-001-002, the cell whose reliability matches Figure 2A:")
print(ex_cell[["probe", "pairs"]].round(3).to_string(index=False))


**Result: reproduced.** One of the 20 candidate probes, probe 4 of 20060105-ha1-001-002 (contexts 2, 3 and silence, 6 trials each), meets the criterion: significant periods at 0.67 to 1.36 s, none from 2 to 4 s, and one from 4.00 to 4.06 s. It was found by searching 20 candidates, so it is a selected example, and its late period is 60 ms long. The cell whose reliability matches the example (20051222-ha1-001-002) has, for probe 4, significant periods up to 1.60 s, none from 1.60 to 3.97 s, and periods again from 3.97 to 4.60 s; the run at 3.97 to 4.01 s starts 30 ms before 4 s, so that probe fails the strict test. Probe 4 is also the probe of that cell whose reliability is 0.737, the 0.74 of Figure 2A, and the Figure 2B legend says the red trace is the context shown in A; so 20051222-ha1-001-002 probe 4 is most likely the paper's example, and the passing probe comes from a cell with mean reliability 0.309, which cannot be it. The claims file's matching rule does not use this link between Figure 2A and 2B and accepts any of the 20 candidates; the outcome follows the criterion as written, and the link is raised as a question for a reader. 17 of 305 natural sound probes have a significant period later than 4 s.

## fig3a-proportion-significant

> Of 305 natural sound probes tested in 39 A1 cells with typically five to eight natural sound
> contexts, 204 (66.9%) showed significant context dependence, and about a quarter (23.7%) of
> the significant events occurred more than 1 s after probe onset.

A probe is significant when the significance measure finds at least one period. The late fraction
is one minus the cumulative probability at 1 s, where the cumulative probability is the running
integral of the proportion curve less the noise floor, normalized at its peak (Figure 3A legend).
Readings: silence counted as a context (main) or not (with 200 shuffles for the noise floor of
the probes whose contexts change when silence is left out, to keep the run short); the proportion at time t over all probes or
over the probes still running.

In [ ]:
LMAX = max(p["T"] for p in nat)
t_full = np.arange(LMAX) / SF

def sig_sets(reading):
    if reading == "with_silence":
        return [(p["signif"], p["noise"]) for p in nat]
    out = []
    for p in nat:
        v = p["nosil"]
        if v == "same":
            out.append((p["signif"], p["noise"]))
        elif v is not None:
            out.append((v["signif"], v["noise"]))
    return out

prop = {}
rows3 = []
for reading in ("with_silence", "without_silence"):
    S = sig_sets(reading)
    nsig = sum(bool(s.any()) for s, _ in S)
    for mode in ("all", "running"):
        P, cnt = asari.pop_curve([s.astype(float) for s, _ in S], mode, LMAX)
        Nf, _ = asari.pop_curve([n.astype(float) for _, n in S], mode, LMAX)
        C = np.cumsum(P - Nf) / SF
        ipk = int(np.argmax(C))
        late = 1 - C[SF] / C[ipk]
        prop[(reading, mode)] = (P, Nf, cnt)
        rows3.append({"silence": reading, "denominator": mode, "probes": len(S), "significant": nsig,
                      "fraction": nsig / len(S), "late_fraction_1s": late, "cum_peak_s": ipk / SF,
                      "mean_noise_floor": Nf.mean()})
r3 = pd.DataFrame(rows3)
print(r3.round(4).to_string(index=False))
print("\nPaper: 204 of 305 (0.669), late fraction 0.237, cumulative peak at 4.72 s, noise floor 2.8e-3.")
for _, r in r3.iterrows():
    print(f"  {r.silence}/{r.denominator}: count within 204 +/- 20.4: {abs(r.significant - 204) <= 20.4}; fraction within 0.669 +/- 0.067: "
          f"{abs(r.fraction - 0.669) <= 0.0669}; late fraction within 0.237 +/- 0.024: {abs(r.late_fraction_1s - 0.237) <= 0.0237}")
avg = [p["signif_avg_ties"] for p in nat]
print(f"\nVariant, average ranks for ties: {sum(bool(s.any()) for s in avg)} of {len(avg)} probes significant")
by_cell = pd.DataFrame([{"cell": p["cell"], "sig": bool(p["signif"].any())} for p in nat]).groupby("cell").sig.agg(["sum", "size"])
print(f"Cells with at least one significant probe: {(by_cell['sum'] > 0).sum()} of {len(by_cell)}; per-cell fraction median {(by_cell['sum'] / by_cell['size']).median():.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
P, Nf, cnt = prop[("with_silence", "running")]
tb = asari.bin_mean(t_full, 100)
ax[0].plot(tb, asari.bin_mean(P, 100), "k", label="proportion significant (running)")
ax[0].plot(tb, asari.bin_mean(prop[("with_silence", "all")][0], 100), "C0", label="proportion (all probes)")
ax[0].plot(tb, asari.bin_mean(Nf, 100), "0.6", label="noise floor")
ax[0].set_xlabel("time from probe onset (s)"); ax[0].legend(fontsize=7)
ax[1].plot(t_full[::100], cnt[::100], "k"); ax[1].set_xlabel("time from probe onset (s)"); ax[1].set_ylabel("probes running")
plt.tight_layout(); plt.show()
print(f"Plotted ranges: proportion (10-ms bins) {asari.bin_mean(P, 100).min():.3f} to {asari.bin_mean(P, 100).max():.3f}; probes running {cnt.min():.0f} to {cnt.max():.0f}")

By animal (recording date and animal code), for the main reading: probes, significant probes and
the mean reliability (pairs of trials).

In [ ]:
animal_of = files.drop_duplicates("cell").set_index("cell").animal
pa = pd.DataFrame([{"animal": animal_of[p["cell"]], "cell": p["cell"], "sig": bool(p["signif"].any()), "rel": p["rel_pairs"]} for p in nat])
by_animal = pa.groupby("animal").agg(cells=("cell", "nunique"), probes=("sig", "size"), significant=("sig", "sum"), reliability=("rel", "mean"))
by_animal["fraction"] = by_animal.significant / by_animal.probes
print(by_animal.round(3).to_string())
print(f"\n{len(by_animal)} animals; fraction significant per animal: median {by_animal.fraction.median():.2f}, range {by_animal.fraction.min():.2f} to {by_animal.fraction.max():.2f}")

**Result: reproduced.** With silence counted as a context, 186 of 305 probes (0.610) are significant (paper 204, 0.669; both within tolerance), and with the proportion taken over all 305 probes the fraction of events later than 1 s is 0.252 (paper 0.237, tolerance 0.024), with the cumulative curve peaking at 4.78 s (paper 4.72 s). Over the probes still running the late fraction is 0.267, outside tolerance. Without silence as a context only 165 probes are significant, outside tolerance. The variant with average ranks for ties gives the same 186.

## fig3a-significance-decay

> The proportion of probes with significant context dependence decays after probe onset as the sum
> of two exponentials (sizes 0.17 and 0.09, decay constants 0.20 and 0.90 s), above a noise floor
> of about 2.8 x 10^-3.

Eq. 44 with the constant fixed at the mean noise floor over the fit range, the number of
exponentials chosen by the paper's rule. Readings: fit range 0 to 4.72 s or 0 to 6 s, with the
readings of the proportion above. Curves are averaged in 1-ms bins for the fit.

In [ ]:
fits3a = []
for (reading, mode), (P, Nf, cnt) in prop.items():
    for tmax in (4.72, 6.0):
        n = int(round(tmax * 1000))
        y = asari.bin_mean(P, 10)[:n]
        tt = (np.arange(len(y)) + 0.5) / 1000
        kappa = float(asari.bin_mean(Nf, 10)[:n].mean())
        chosen, allf = asari.fit_paper_rule(tt, y, kappa=kappa, nonneg=True)
        two = allf[1]
        fits3a.append({"silence": reading, "denominator": mode, "fit_to_s": tmax, "kappa": kappa, "n_chosen": chosen["n"],
                       "alpha1": two["alpha"][0], "tau1": two["tau"][0], "alpha2": two["alpha"][1], "tau2": two["tau"][1],
                       "one_exp_alpha": allf[0]["alpha"][0], "one_exp_tau": allf[0]["tau"][0]})
f3a = pd.DataFrame(fits3a)
exp_ = {"alpha1": (0.17, 0.017), "tau1": (0.20, 0.02), "alpha2": (0.09, 0.009), "tau2": (0.90, 0.09), "kappa": (0.0028, 0.00028)}
for k, (v, tol) in exp_.items():
    f3a["ok_" + k] = (f3a[k] - v).abs() <= tol
print(f3a.round(4).to_string(index=False))
print("\nPaper: two exponentials, alpha1 0.17, tau1 0.20 s, alpha2 0.09, tau2 0.90 s, kappa 2.8e-3. The two-exponential fit is shown whatever the rule chose.")

**Result: partially reproduced.** Two exponentials are selected under every reading with silence counted, with a slow constant of 0.97 to 1.08 s. Over all probes the fit to 4.72 s gives 0.149, 0.252 s, 0.079 and 0.968 s with a mean noise floor of 2.5e-3; only the slow constant is within tolerance (paper 0.17, 0.20 s, 0.09, 0.90 s, 2.8e-3). Over the running probes the fast size (0.159) and the noise floor (2.8e-3) are within tolerance and the slow constant (1.075 s) is not. The fast constant is 0.25 to 0.27 s under every reading, against 0.20 s.

## fig3b-fractional-power-decay

> The context-dependent fraction of the stimulus-related response power, normalized by the
> smoothed stimulus-related power, decays from about 0.49 with a time constant of about 1.04 s.

Q(t) of Eq. 20: the population averages of Eq. 19 (predictable less context-independent power)
over the population average of Eq. 17 smoothed over [0, 2t]; one exponential with constant 0.
Readings: silence as a context or not, probes averaged over all or over those running, fit range 0
to 4 s or 0 to 6 s.

In [ ]:
def pow_sets(reading):
    if reading == "with_silence":
        return [(p["pred"], p["inv"]) for p in nat]
    out = []
    for p in nat:
        v = p["nosil"]
        if v == "same":
            out.append((p["pred"], p["inv"]))
        elif v is not None:
            out.append((v["pred"], v["inv"]))
    return out

curves = {}
rows3b = []
for reading in ("with_silence", "without_silence"):
    S = pow_sets(reading)
    for mode in ("all", "running"):
        pred, cnt = asari.pop_curve([a.astype(float) for a, _ in S], mode, LMAX)
        inv, _ = asari.pop_curve([b.astype(float) for _, b in S], mode, LMAX)
        den = asari.running_mean_0_2t(pred)
        Q = (pred - inv) / den
        U = inv / den
        U_raw = inv / pred
        curves[(reading, mode)] = (pred, inv, den, Q, U, U_raw)
        for tmax in (4.0, 6.0):
            y = asari.bin_mean(Q, 10)[:int(tmax * 1000)]
            fq = asari.fit_exp((np.arange(len(y)) + 0.5) / 1000, y, 1, kappa=0.0)
            rows3b.append({"silence": reading, "denominator": mode, "fit_to_s": tmax, "alpha1": fq["alpha"][0], "tau1": fq["tau"][0],
                           "Q_mean_0_100ms": Q[:1000].mean(), "Q_mean_3_4s": Q[30000:40000].mean()})
r3b = pd.DataFrame(rows3b)
r3b["ok_alpha1"] = (r3b.alpha1 - 0.49).abs() <= 0.049
r3b["ok_tau1"] = (r3b.tau1 - 1.04).abs() <= 0.104
print(r3b.round(3).to_string(index=False))
print("\nPaper: alpha1 0.49, tau1 1.04 s.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
pred, inv, den, Q, U, U_raw = curves[("with_silence", "running")]
ax[0].plot(tb, asari.bin_mean(pred, 100), "k", lw=0.8, label="stimulus-related (Eq. 17)")
ax[0].plot(tb, asari.bin_mean(den, 100), "0.6", label="running average [0, 2t]")
ax[0].plot(tb, asari.bin_mean(pred - inv, 100), "C3", label="context-dependent (Eq. 19)")
ax[0].set_ylabel("mV^2"); ax[0].legend(fontsize=7); ax[0].set_xlabel("time from probe onset (s)")
ax[1].plot(tb, asari.bin_mean(Q, 100), "k", label="Q(t), Eq. 20")
ax[1].plot(tb, asari.bin_mean(U, 100), "C0", label="Eq. 43")
ax[1].set_ylim(-0.2, 1.4); ax[1].legend(fontsize=7); ax[1].set_xlabel("time from probe onset (s)")
plt.tight_layout(); plt.show()
qb, ub = asari.bin_mean(Q, 100), asari.bin_mean(U, 100)
print(f"Ranges (10-ms bins): Q {qb.min():.2f} to {qb.max():.2f}, Eq. 43 {ub.min():.2f} to {ub.max():.2f}; "
      f"bins outside the right axis: {int(((qb < -0.2) | (qb > 1.4)).sum() + ((ub < -0.2) | (ub > 1.4)).sum())}")

**Result: not reproduced.** The size of the decay is within tolerance under every reading (0.445 to 0.468, paper 0.49), but the time constant is 1.74 to 1.78 s with silence counted and 1.29 to 1.33 s without it, against 1.04 s (tolerance 0.104 s), so no reading has both within tolerance, which the criterion requires; it gives no rule for partial. The normalized context-dependent power decays on a timescale of seconds, more slowly than the paper reports.

## timescale-longer-than-membrane

> The trial-to-trial noise autocorrelation and the autocorrelation of spontaneous activity decay
> within about 100 ms, much faster than the second-long timescale of context dependence in A1.

Noise: each trial less its context mean (Eq. 14), autocorrelation normalized at lag 0 (Eq. 45),
averaged over probes. Spontaneous: the intersequence silences, whole and without their first 1 s.
The lag where the mean autocorrelation first falls below 1/e, and below 0.1 (variant).

In [ ]:
lags = np.arange(asari.AC_MAX_LAG // asari.AC_STEP + 1) * asari.AC_STEP / SF
def cross(ac, level):
    i = np.where(ac < level)[0]
    return lags[i[0]] if len(i) else np.nan
noise_ac = np.mean([p["noise_ac"] / p["noise_ac"][0] for p in nat], axis=0)
sp_whole = [r["spont_ac"]["whole"] for r in results if r.get("spont_ac") and r["spont_ac"]["whole"] is not None]
sp_after = [r["spont_ac"]["after_1s"] for r in results if r.get("spont_ac") and r["spont_ac"]["after_1s"] is not None]
spw = np.mean([a / a[0] for a in sp_whole], axis=0)
spa = np.mean([a / a[0] for a in sp_after], axis=0)
tab = pd.DataFrame([{"signal": n, "lag_1/e_ms": 1000 * cross(a, 1 / np.e), "lag_0.1_ms": 1000 * cross(a, 0.1),
                     "ac_at_100ms": a[100], "ac_at_500ms": a[500]} for n, a in
                    (("trial-to-trial noise", noise_ac), ("spontaneous, whole silence", spw), ("spontaneous, after first 1 s", spa))])
print(tab.round(3).to_string(index=False))
per_probe = np.array([cross(p["noise_ac"] / p["noise_ac"][0], 1 / np.e) for p in nat]) * 1000
print(f"\nPer-probe 1/e lag of the noise autocorrelation (ms): median {np.nanmedian(per_probe):.1f}, 90th percentile {np.nanpercentile(per_probe, 90):.1f}, max {np.nanmax(per_probe):.1f}")
print(f"Files with silences: {len(sp_whole)}; one third of 1.04 s is 347 ms; the criterion's bound is 150 ms.")

**Result: reproduced.** The mean autocorrelation of the trial-to-trial noise falls below 1/e at 73 ms (below 0.1 at 116 ms), and that of the spontaneous activity in the intersequence silences at 78 ms (77 ms without the first 1 s). Both 1/e lags are under 150 ms and under a third of 1.04 s.

## fig4-predictability-upper-bound

> Without knowledge of the context, no model can capture more than about half (0.51) of the
> predictable response power with a window of about 100 ms; the upper bound approaches one only
> with a window of seconds (time constant 1.04 s).

Eq. 43: the context-independent power over the stimulus-related power, smoothed over [0, 2t]
(main) or not (variant); one exponential with the constant fixed at 1, so the bound near onset is
1 + alpha1.

In [ ]:
rows4 = []
for (reading, mode), (pred, inv, den, Q, U, U_raw) in curves.items():
    for smooth, curve in (("smoothed", U), ("unsmoothed", U_raw)):
        for tmax in (4.0, 6.0):
            y = asari.bin_mean(curve, 10)[:int(tmax * 1000)]
            f = asari.fit_exp((np.arange(len(y)) + 0.5) / 1000, y, 1, kappa=1.0)
            rows4.append({"silence": reading, "denominator": mode, "denominator_smoothing": smooth, "fit_to_s": tmax,
                          "1+alpha1": 1 + f["alpha"][0], "tau1": f["tau"][0], "mean_0_100ms": curve[:1000].mean()})
r4 = pd.DataFrame(rows4)
r4["ok_bound"] = (r4["1+alpha1"] - 0.51).abs() <= 0.051
r4["ok_tau"] = (r4.tau1 - 1.04).abs() <= 0.104
r4["tau_at_bound"] = r4.tau1 >= 99
print(r4.round(3).to_string(index=False))
print("\nPaper: 1 + alpha1 = 0.51, tau1 1.04 s.")
print(f"Fits with the time constant at the 100 s bound of the fit: {int(r4.tau_at_bound.sum())} of {len(r4)}")
for (reading, mode), (pred, inv, den, Q, U, U_raw) in curves.items():
    ub = asari.bin_mean(U, 10)
    print(f"  {reading}/{mode}: smoothed Eq. 43 (1-ms bins) ranges {np.nanmin(ub):.2f} to {np.nanmax(ub):.2f}; 1-ms bins above 1 within the first 6 s: {int((ub[:6000] > 1).sum())}")


**Result: not reproduced.** Near probe onset the upper bound is about half: 1 + alpha1 is 0.50 to 0.54 with the unsmoothed ratio (paper 0.51) and 0.43 to 0.49 with the smoothed denominator, and the mean over the first 100 ms is 0.50 (smoothed, silence counted; 0.57 without silence) or 0.39 (unsmoothed; 0.44 without silence). Two fits to 6 s over all probes (smoothed) ran to the 100 s bound of the time constant and are not counted; the smoothed ratio exceeds 1 in some bins (maximum 1.67 and 1.83 over all probes, 2.16 and 2.61 over the probes running). The approach to one is slower than in the paper: the time constant is 1.46 to 1.92 s (unsmoothed) and 1.66 to 2.81 s (smoothed, fits not at the bound) against 1.04 s, outside tolerance under every reading, so the criterion's joint test fails; it gives no rule for partial.

## Synthetic conditioning stimuli (Figure 8)

The paper tests each sound property with only the conditioning stimuli that differ in it
(Methods, Stimulus design, paragraph 2), but does not list, for each recording, which stimuli form
a set. The sets are rebuilt from each recording's stimulus metadata (`asari.property_families`);
this is an assumption the Figure 8 outcomes rest on. A probe can be tested with more than one set
of one property: the 16 modulated harmonic tones of the 17-stimulus recordings are a grid of 4 AM
rates by 4 FM rates, so the natural sound probe meets four AM sets and four FM sets, and forward
and time-reversed DMRs and TORCs form separate sets. Three readings are run: one test per probe
and property with its largest set; one test per set; and one test per set for AM and FM with the
largest set for the other properties. The last reading gives the totals of Table 1 for all five
properties (95, 110, 96, 82 and 137) and is the main one; the others are shown. The notes, stimuli
and rebuilt sets of every MHT recording are printed below.

In [ ]:
SY_ALL = [{**s, "cell": r["cell"], "file": r["file"]} for r in results for s in r.get("synthetic", [])]
READINGS = {"largest set": lambda s: s["largest"], "every set": lambda s: True,
            "every set for AM and FM": lambda s: s["largest"] or s["property"] in ("AM", "FM")}
MAIN = "every set for AM and FM"
SYr = {k: [s for s in SY_ALL if f(s)] for k, f in READINGS.items()}
SY = SYr[MAIN]
def frame(S):
    d = pd.DataFrame([{k: v for k, v in s.items() if k not in ("signif", "noise", "pred", "inv")} | {"sig": bool(s["signif"].any())} for s in S])
    d["row"] = np.where(d.probe_type == "natural", "natural", d.ensemble)
    return d
syr = {k: frame(S) for k, S in SYr.items()}
sy = syr[MAIN]
totals8 = {"amplitude": 95, "frequency": 110, "AM": 96, "FM": 82, "higher-order": 137}
print("Probe tests per property under each reading of the property sets, with the paper's totals (Table 1):")
cmp8 = pd.DataFrame({k: d.groupby("property").size() for k, d in syr.items()}).reindex(asari.PROPERTIES)
cmp8["paper"] = pd.Series(totals8)
print(cmp8.to_string())
print("Readings matching every total:", [k for k in syr if (cmp8[k] == cmp8.paper).all()])
table1 = {("amplitude", "natural"): 23, ("frequency", "natural"): 25, ("AM", "natural"): 25, ("FM", "natural"): 25, ("higher-order", "natural"): 63,
          ("amplitude", "torc"): 39, ("frequency", "torc"): 40, ("amplitude", "dmr"): 8, ("frequency", "dmr"): 20,
          ("amplitude", "synthetic"): 25, ("frequency", "synthetic"): 25, ("AM", "synthetic"): 71, ("FM", "synthetic"): 57, ("higher-order", "pinknoise"): 74}
t1 = sy.groupby(["property", "row"]).agg(probes=("probe", "size"), cells=("cell", "nunique"), significant=("sig", "sum")).reset_index()
t1["table1_probes"] = [table1.get((p, r), np.nan) for p, r in zip(t1.property, t1.row)]
print(f"\nMain reading ({MAIN}), by probe type (row: natural probe, or the ensemble of a synthetic probe; 'synthetic' is MHT):")
print(t1.to_string(index=False))

print("\nModulated harmonic tone recordings (trigger type 'synthetic'), grouped by identical stimuli and notes:")
groups = {}
for r in results:
    if r.get("stimuli") is not None and r["info"]["trigger_types"] == ["synthetic"]:
        tabs = pd.DataFrame(r["stimuli"])
        key = (tabs.drop(columns="name").astype(str).to_csv(index=False), tuple(l for l in r["notes_lines"] if not l.startswith(("Effective", "Natural", "The Diversity", "Sounds of"))))
        groups.setdefault(key, []).append(r)
for (tabtxt, notes), rs in groups.items():
    print(f"\n=== {len(rs)} recordings: " + ", ".join(f'{x["file"].split("/")[-1]} (rep {x["info"]["rep"]})' for x in rs))
    print("notes:", " | ".join(notes) if notes else "-")
    cols = ["stim", "name", "description", "f0", "AMmean", "AMstd", "AMrange", "FMrange", "FMstd"]
    tabs = pd.DataFrame(rs[0]["stimuli"])
    print(tabs[[c for c in cols if c in tabs]].to_string(index=False))
    print("sets rebuilt:", {k: v for k, v in rs[0]["families"].items() if v})
    n_tests = {}
    for x in rs:
        for t in x.get("synthetic", []):
            n_tests[(t["property"], t["probe_type"])] = n_tests.get((t["property"], t["probe_type"]), 0) + 1
    print("tests (every set) by property and probe type over these recordings:", n_tests)

## fig8a-proportion-by-property

> When only one property of synthetic conditioning stimuli varied, context dependence was found in
> 77 of 95 probes (81.1%, 31 cells) for amplitude, 73 of 110 (35 cells) for frequency, 62 of 96
> (64.6%, 27 cells) for AM, 47 of 82 (57.3%, 25 cells) for FM and 59 of 137 (43.1%, 27 cells) for
> higher-order properties.

In [ ]:
paper8 = {"amplitude": (77, 95, 31), "frequency": (73, 110, 35), "AM": (62, 96, 27), "FM": (47, 82, 25), "higher-order": (59, 137, 27)}
r8s = {}
for k, d in syr.items():
    r8 = d.groupby("property").agg(probes=("probe", "size"), cells=("cell", "nunique"), significant=("sig", "sum")).reindex(asari.PROPERTIES)
    r8["fraction"] = r8.significant / r8.probes
    r8["paper_significant"] = [paper8[p][0] for p in r8.index]
    r8["paper_probes"] = [paper8[p][1] for p in r8.index]
    r8["paper_fraction"] = r8.paper_significant / r8.paper_probes
    r8["count_within_10pct"] = (r8.significant - r8.paper_significant).abs() <= 0.1 * r8.paper_significant
    r8s[k] = r8
    fr = r8.fraction
    print(f"--- {k}{' (main reading)' if k == MAIN else ''}")
    print(r8.round(3).to_string())
    print(f"Order amplitude > frequency: {fr['amplitude'] > fr['frequency']}; AM > FM > higher-order: {fr['AM'] > fr['FM'] > fr['higher-order']}; "
          f"counts within tolerance: {int(r8.count_within_10pct.sum())} of 5\n")

**Result: partially reproduced.** Under the main reading (whose denominators equal Table 1's 95, 110, 96, 82 and 137) the significant probes are 56 (amplitude), 62 (frequency), 41 (AM), 29 (FM) and 45 (higher-order), all outside tolerance of the paper's 77, 73, 62, 47 and 59. The proportions keep the paper's order: amplitude 0.589 > frequency 0.564, and AM 0.427 > FM 0.354 > higher-order 0.328. The order holds under the other two readings of the property sets as well, and no reading has more than one count within tolerance. The outcome rests on the rebuilt property sets.

## fig8a-significance-fits

> The proportion of probes with significant context dependence decays fastest and is smallest for
> higher-order changes, with fitted (size, time constant) of (0.23, 0.25 s) for amplitude, (0.19,
> 0.27 s) for frequency, (0.13, 0.17 s) for AM, (0.07, 0.15 s) for FM and (0.04, 0.13 s) for
> higher-order properties.

One exponential with the constant fixed at the property's mean noise floor, fitted from 0 to 4 s;
the proportion over all probes of the property (main) or over those running.

In [ ]:
paper8a = {"amplitude": (0.23, 0.25, 2.8e-3), "frequency": (0.19, 0.27, 2.3e-3), "AM": (0.13, 0.17, 2.3e-3), "FM": (0.07, 0.15, 2.1e-3), "higher-order": (0.04, 0.13, 3.6e-3)}
rows8a = []
for reading, S_r in SYr.items():
    for prop in asari.PROPERTIES:
        S = [s for s in S_r if s["property"] == prop]
        L = max(s["T"] for s in S)
        for mode in ("all", "running"):
            P, cnt = asari.pop_curve([s["signif"].astype(float) for s in S], mode, L)
            Nf, _ = asari.pop_curve([s["noise"].astype(float) for s in S], mode, L)
            y = asari.bin_mean(P, 10)[:4000]
            kappa = float(asari.bin_mean(Nf, 10)[:4000].mean())
            f = asari.fit_exp((np.arange(len(y)) + 0.5) / 1000, y, 1, kappa=kappa, nonneg=True)
            a, t, k = paper8a[prop]
            rows8a.append({"reading": reading, "property": prop, "denominator": mode, "probes": len(S), "alpha1": f["alpha"][0], "tau1": f["tau"][0], "kappa": kappa,
                           "paper_alpha1": a, "paper_tau1": t, "paper_kappa": k, "ok_alpha1": abs(f["alpha"][0] - a) <= 0.1 * a, "ok_tau1": abs(f["tau"][0] - t) <= 0.1 * t})
r8a = pd.DataFrame(rows8a)
print(r8a.round(4).to_string(index=False))
for (reading, mode), g in r8a.groupby(["reading", "denominator"], sort=False):
    al = g.set_index("property").alpha1
    print(f"{reading}, {mode}: sizes in the printed order amplitude > frequency > AM > FM > higher-order: {bool(np.all(np.diff(al.values) < 0))}; "
          f"sizes within tolerance: {int(g.ok_alpha1.sum())} of 5; time constants within tolerance: {int(g.ok_tau1.sum())} of 5")

**Result: not reproduced.** Under the main reading the fitted sizes are 0.124, 0.147, 0.065, 0.012 and 0.025 (paper 0.23, 0.19, 0.13, 0.07, 0.04), none within tolerance, and they do not fall in the printed order (amplitude is below frequency, FM below higher-order). The time constants of frequency (0.286 s) and higher-order (0.129 s) are within tolerance; amplitude (0.279 s), AM (0.199 s) and FM (0.134 s) are not. No reading of the property sets has a size within tolerance or the sizes in the printed order. The outcome rests on the rebuilt property sets.

## fig8b-decay-by-property

> The normalized context-dependent power decays slowly for amplitude and frequency changes (time
> constants 0.66 and 0.97 s) and fast for AM, FM and higher-order changes (0.11, 0.15 and 0.20 s),
> so higher-order properties act mainly on a timescale of about 100 ms.

Q(t) of Eq. 20 per property, one exponential with constant 0, fitted from 0 to 4 s (main) and from
0 to the shortest probe of the property (variant); probes averaged over those running (main) or
over all.

In [ ]:
paper8b = {"amplitude": (0.57, 0.66), "frequency": (0.54, 0.97), "AM": (0.86, 0.11), "FM": (0.64, 0.15), "higher-order": (0.52, 0.20)}
rows8b, Qprop = [], {}
for reading, S_r in SYr.items():
    for prop in asari.PROPERTIES:
        S = [s for s in S_r if s["property"] == prop]
        L = max(s["T"] for s in S)
        short = min(s["T"] for s in S) / SF
        for mode in ("running", "all"):
            pred, _ = asari.pop_curve([s["pred"].astype(float) for s in S], mode, L)
            inv, _ = asari.pop_curve([s["inv"].astype(float) for s in S], mode, L)
            Q = (pred - inv) / asari.running_mean_0_2t(pred)
            Qprop[(reading, prop, mode)] = Q
            for rng_name, tmax in (("0-4 s", 4.0), ("0-shortest", short)):
                y = asari.bin_mean(Q, 10)[:int(tmax * 1000)]
                f = asari.fit_exp((np.arange(len(y)) + 0.5) / 1000, y, 1, kappa=0.0)
                a, t = paper8b[prop]
                rows8b.append({"reading": reading, "property": prop, "denominator": mode, "fit": rng_name, "fit_to_s": round(tmax, 2), "alpha1": f["alpha"][0], "tau1": f["tau"][0],
                               "paper_alpha1": a, "paper_tau1": t, "ok_tau1": abs(f["tau"][0] - t) <= 0.1 * t, "ok_alpha1": abs(f["alpha"][0] - a) <= 0.1 * a})
r8b = pd.DataFrame(rows8b)
print(r8b.round(3).to_string(index=False))
for (reading, mode, fit), g in r8b.groupby(["reading", "denominator", "fit"], sort=False):
    tau = g.set_index("property").tau1
    print(f"{reading}, {mode}, {fit}: time constants within tolerance {int(g.ok_tau1.sum())} of 5; amplitude and frequency > 0.4 s: {bool(tau['amplitude'] > 0.4 and tau['frequency'] > 0.4)}; "
          f"AM, FM, higher-order < 0.3 s: {bool((tau[['AM', 'FM', 'higher-order']] < 0.3).all())}")

**Result: partially reproduced.** Under the main reading, with the probes running and the fit to 4 s, the time constants are 1.40 s (amplitude), 0.87 s (frequency), 0.109 s (AM), 0.278 s (FM) and 0.186 s (higher-order) (paper 0.66, 0.97, 0.11, 0.15, 0.20 s). Frequency, AM and higher-order are within tolerance and amplitude and FM are not. Amplitude and frequency are both above 0.4 s and AM, FM and higher-order below 0.3 s under every reading of the denominator, fit range and property sets, which meets the rule for partial; the FM time constant of the main reading (0.278 s) is close to that 0.3 s line. The outcome rests on the rebuilt property sets.

## fig8c-relative-effects

> Changes in amplitude and frequency of the conditioning stimuli produced context effects as large
> and long-lasting as natural sound contexts, while AM, FM and higher-order changes produced
> substantially smaller and shorter effects.

The area under Q(t) from 0 to 4 s per property over the area for natural sounds (Figure 3B), with
95% intervals from 200 resamplings of 1000 probes drawn with replacement. The natural sound
reference is run with silence counted as a context (the Figure 3B main reading) and without it
(the synthetic property sets never include silence), for each reading of the property sets. The
other reading of a "sample" (trials within probes) needs the single trials, which the pass does not
keep; it is not run.

In [ ]:
rng8c = np.random.default_rng(8)
L4 = 40000
def stack(S):
    P = np.zeros((len(S), L4)); I = np.zeros((len(S), L4)); M = np.zeros((len(S), L4))
    for j, (a, b) in enumerate(S):
        n = min(len(a), L4); P[j, :n] = a[:n]; I[j, :n] = b[:n]; M[j, :n] = 1
    return P, I, M
def q_area(st, w):
    P, I, M = st
    c = np.maximum(w @ M, 1)
    pred, inv = (w @ P) / c, (w @ I) / c      # probes running at t, weighted by how often drawn
    Q = (pred - inv) / asari.running_mean_0_2t(pred)
    return Q.sum() / SF
def boot(st, n):
    return np.array([q_area(st, np.bincount(rng8c.integers(0, n, 1000), minlength=n).astype(float)) for _ in range(200)])
nat_ref = {}
for ref in ("with_silence", "without_silence"):
    S = [(a.astype(float), b.astype(float)) for a, b in pow_sets(ref)]
    st = stack(S)
    nat_ref[ref] = (q_area(st, np.ones(len(S))), boot(st, len(S)), len(S))
    a0, bt, n = nat_ref[ref]
    print(f"Natural sounds, {ref}: {n} probes, area 0-4 s {a0:.3f} s; resampled 95% interval {np.percentile(bt, 2.5):.3f} to {np.percentile(bt, 97.5):.3f}")
prop_boot = {}
for reading, S_r in SYr.items():
    for prop in asari.PROPERTIES:
        S = [(s["pred"].astype(float), s["inv"].astype(float)) for s in S_r if s["property"] == prop]
        st = stack(S)
        prop_boot[(reading, prop)] = (q_area(st, np.ones(len(S))), boot(st, len(S)), len(S))
# Trial reading: trials drawn with replacement within each probe and context (asari.frac_power_trial_boot, in the pass),
# 200 resamplings, the curves in 20-ms bins over the first 4 s; the area of each draw is formed as q_area forms it.
BB = asari.BOOT_BIN
def trial_areas(boots):
    P = np.stack([b[0] for b in boots]).astype(float); I = np.stack([b[1] for b in boots]).astype(float)
    M = np.zeros((len(boots), P.shape[2]))
    for j, b in enumerate(boots):
        M[j, :b[2]] = 1
    c = np.maximum(M.sum(0), 1)
    pred, inv = P.sum(0) / c, I.sum(0) / c                                    # draws x bins, probes running
    den = np.array([asari.running_mean_0_2t(v) for v in pred])
    return ((pred - inv) / den).sum(1) * BB / SF
def nat_boots(ref):
    if ref == "with_silence":
        return [p["boot"] for p in nat]
    return [p["boot"] if p["nosil"] == "same" else p["nosil"]["boot"] for p in nat if p["nosil"] is not None]
nat_trial = {ref: trial_areas(nat_boots(ref)) for ref in nat_ref}
for ref, bt in nat_trial.items():
    print(f"Natural sounds, {ref}, trial reading: resampled area 0-4 s median {np.median(bt):.3f} s, 95% interval {np.percentile(bt, 2.5):.3f} to {np.percentile(bt, 97.5):.3f}")
prop_trial = {(reading, prop): trial_areas([s["boot"] for s in S_r if s["property"] == prop]) for reading, S_r in SYr.items() for prop in asari.PROPERTIES}

rows8c = []
for sample in ("probes", "trials"):
    for reading in SYr:
        for ref, (a0, bt0, _) in nat_ref.items():
            for prop in asari.PROPERTIES:
                area, bt, n = prop_boot[(reading, prop)]
                ratio = bt / bt0 if sample == "probes" else prop_trial[(reading, prop)] / nat_trial[ref]
                rows8c.append({"sample": sample, "reading": reading, "natural_reference": ref, "property": prop, "probes": n, "area_0_4s": area,
                               "ratio": area / a0, "resampled_median": np.median(ratio),
                               "ci_low": np.percentile(ratio, 2.5), "ci_high": np.percentile(ratio, 97.5)})
r8c_all = pd.DataFrame(rows8c)
print(r8c_all.round(3).to_string(index=False))
print()
for (sample, reading, ref), g in r8c_all.groupby(["sample", "reading", "natural_reference"], sort=False):
    g = g.set_index("property")
    lo, hi = g.loc[["amplitude", "frequency"]], g.loc[["AM", "FM", "higher-order"]]
    ok_low = bool((lo.ci_high >= 1).all())
    ok_high = bool((hi.ci_high < 1).all() and (hi.ci_high < lo.ci_low.min()).all())
    pt_ok = bool((lo.ratio >= 1).all() and (hi.ratio < min(1, lo.ratio.min())).all())
    pt_lower = bool((hi.ratio < min(1, lo.ratio.min())).all())
    print(f"{sample} resampled, {reading}, natural reference {ref}: amplitude and frequency intervals include or exceed 1: {ok_low}; AM, FM, higher-order intervals below 1 and below the "
          f"amplitude and frequency lower bounds: {ok_high}; point estimates in the pattern (amplitude and frequency at least 1, the others below 1 and below them): {pt_ok}; "
          f"AM, FM, higher-order point estimates below 1 and below amplitude and frequency: {pt_lower}")
r8c = r8c_all[(r8c_all.sample == "probes") & (r8c_all.reading == MAIN) & (r8c_all.natural_reference == "with_silence")].set_index("property")

**Result: not reproduced.** Under the main reading with the natural reference that counts silence, the areas relative to natural sounds are 0.72 (amplitude, interval 0.64 to 0.85), 0.65 (frequency, 0.57 to 0.76), 0.15 (AM), 0.13 (FM) and 0.51 (higher-order). With the natural reference that leaves silence out they are 0.89 (amplitude, 0.75 to 1.06), 0.80 (frequency, 0.70 to 0.93), 0.18, 0.16 and 0.63 (higher-order, up to 0.77). Under no reading do the intervals of both amplitude and frequency include 1, and the amplitude and frequency point estimates are below 1 in every reading, so neither the rule for reproduced nor the rule for partial (point estimates in the pattern) is met. AM, FM and higher-order are below 1 and below amplitude and frequency in every reading, so the lower-order against higher-order part of the claim holds. In the trial reading (trials drawn with replacement within each probe and context, 20-ms bins) the intervals of amplitude and frequency lie above 1 under every reading of the sets and both references (lowest bound 1.027), and those of AM and FM lie below 1 and below them except AM with the largest set and silence left out (0.829 to 1.010), but the upper bound for higher-order is 1.003 to 1.078, so the rule for reproduced is not met under any reading either (closest: every set, silence counted, higher-order 0.909 to 1.003). The trial-resampled areas are not centred on the measured ones: for natural sounds the median is 1.69 s against 0.752 s. Drawing a trial twice adds its noise to the predictable power of Eq. 17, which assumes distinct trials, and this pulls every ratio towards 1. The point estimates are the same in both readings, so the rule for partial is not met. The outcome rests on the rebuilt property sets.

## fig8-natural-synthetic-probes-alike

> Natural and synthetic probes gave no difference in the context effects of the synthetic
> conditioning stimuli, so they were combined.

Per property, the proportion of significant probes, natural against synthetic probes, by Fisher's
exact test at 0.05 with a Bonferroni correction over the properties tested with both (the
claims file's test).

In [ ]:
for reading, d in syr.items():
    rowsf = []
    for prop in asari.PROPERTIES:
        g = d[d.property == prop]
        a = g[g.probe_type == "natural"].sig
        b = g[g.probe_type == "synthetic"].sig
        if len(a) and len(b):
            tab = [[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]]
            o, p = stats.fisher_exact(tab)
            rowsf.append({"property": prop, "natural": f"{a.sum()}/{len(a)}", "synthetic": f"{b.sum()}/{len(b)}", "synthetic_cells": g[g.probe_type == "synthetic"].cell.nunique(),
                          "frac_natural": a.mean(), "frac_synthetic": b.mean(), "odds_ratio": o, "p": p})
        else:
            rowsf.append({"property": prop, "natural": f"{a.sum()}/{len(a)}", "synthetic": f"{b.sum()}/{len(b)}", "synthetic_cells": g[g.probe_type == "synthetic"].cell.nunique(),
                          "frac_natural": a.mean() if len(a) else np.nan, "frac_synthetic": b.mean() if len(b) else np.nan, "odds_ratio": np.nan, "p": np.nan})
    rf = pd.DataFrame(rowsf)
    k = rf.p.notna().sum()
    rf["p_bonferroni"] = (rf.p * k).clip(upper=1)
    print(f"--- {reading}{' (main reading)' if reading == MAIN else ''}")
    print(rf.round(4).to_string(index=False))
    print(f"Properties tested with both probe types: {k}; any difference at 0.05 after Bonferroni: {bool((rf.p_bonferroni < 0.05).any())}\n")

**Result: reproduced.** Under the main reading Fisher's exact test finds no difference between natural and synthetic probes for any of the four properties tested with both (amplitude 0.55 against 0.60, frequency 0.57 against 0.56, AM 0.40 against 0.57, higher-order 0.33 against 0.33; Bonferroni-corrected p = 1 for each), and none under the other readings (smallest corrected p 0.83). No synthetic probe was formed for FM, and the AM synthetic probes (14) come from 2 cells, so the test has little power for AM and none for FM. The test is the claims file's, not the paper's, and the property sets are rebuilt from the metadata.

## fig9-mgb-fast-decay

> In the auditory thalamus (MGB), context dependence of spike responses decayed within a short
> time (time constant about 80 ms; 93 probes in 14 cells), so the long-lasting component in A1
> arises mainly in cortex.

PSTHs in 100-ms bins per context, the SD over contexts (n - 1), averaged over probes for k = 0 to
60; one exponential with a free constant. Readings: bin time at its start or its centre; probes
averaged over all or over those running. The last bin of `histc` (spikes exactly at the probe's
end) has no width and is left out. Over the probes running, a bin where no probe runs (bin 60) is
left out of the average and the fit.

In [ ]:
print(f"MGB probes: {len(MGB)} in {len({p['cell'] for p in MGB})} cells; trials per context {min(p['rep'] for p in MGB)} to {max(p['rep'] for p in MGB)}")
sds = [p["sd"][:-1] for p in MGB]
rows9 = []
for mode in ("all", "running"):
    m9, cnt9 = asari.pop_curve(sds, mode, 61)
    for when in ("start", "centre"):
        tt = np.arange(61) * 0.1 + (0.05 if when == "centre" else 0.0)
        f = asari.fit_exp(tt, m9, 1)
        rows9.append({"denominator": mode, "bin_time": when, "alpha1": f["alpha"][0], "tau1_s": f["tau"][0], "kappa": f["kappa"],
                      "mean_sd_bin0": m9[0], "mean_sd_after_0.5s": np.nanmean(m9[5:]), "plateau_within_25pct": abs(np.nanmean(m9[5:]) - f["kappa"]) <= 0.25 * f["kappa"]})
r9 = pd.DataFrame(rows9)
print(r9.round(3).to_string(index=False))
print("\nPaper: alpha1 15.0 spike/s, tau1 0.080 s, kappa 2.0 spike/s.")
m9, cnt9 = asari.pop_curve(sds, "running", 61)
print("Mean SD of PSTHs, first 8 bins (running):", m9[:8].round(2).tolist(), "; probes running in bins 0, 40, 50, 60:", cnt9[[0, 40, 50, 60]].tolist(), "(bins where no probe runs are left out of the running average and its fit)")
animal_m = files.drop_duplicates("cell").set_index("cell").animal
pm = pd.DataFrame([{"animal": animal_m[p["cell"]], "b0": p["sd"][0], "late": p["sd"][5:-1].mean()} for p in MGB])
print("By animal (probes, mean SD in bin 0 and after 0.5 s):")
print(pm.groupby("animal").agg(probes=("b0", "size"), bin0=("b0", "mean"), after_05s=("late", "mean")).round(2).to_string())
for an, g in [(a, [p for p in MGB if animal_m[p["cell"]] == a]) for a in sorted(pm.animal.unique())]:
    m_a, _ = asari.pop_curve([p["sd"][:-1] for p in g], "all", 61)
    f_a = asari.fit_exp(np.arange(61) * 0.1, m_a, 1)
    print(f"  {an}: {len(g)} probes, tau1 {f_a['tau'][0]:.3f} s, alpha1 {f_a['alpha'][0]:.2f}, kappa {f_a['kappa']:.2f}")
print("Per cell, mean SD in bin 0 and after 0.5 s:")
print(pd.DataFrame([{"cell": p["cell"], "b0": p["sd"][0], "late": p["sd"][5:-1].mean()} for p in MGB]).groupby("cell").mean().round(2).to_string())

**Result: reproduced.** The mean SD of the PSTHs over contexts decays with a time constant of 0.080 s over all 93 probes (0.055 s over the probes running), and after 0.5 s it stays at 0.79 spike/s, within 25% of the fitted constant (0.81). The sizes are smaller than the paper's: alpha1 5.6 (bin start) or 10.4 (bin centre) against 15.0 spike/s, and kappa 0.81 against 2.0 spike/s. By animal the time constant is 0.080, 0.086 and 0.070 s in three of the four animals; in the fourth (23 probes) the fit does not find a decay (tau1 at the 100 s bound).

## fig9-example-first-bin

> In an example MGB unit, the response to a probe differed between contexts only in the first
> 100-ms bin of the PSTH.

Among MGB probes whose contexts are silence and two fragments, those whose SD of PSTHs over contexts
is larger in the first bin than in every later bin by at least a factor of two; and the fraction of
all MGB probes for which this holds.

In [ ]:
def first_bin(p):
    sd = p["sd"][:-1]
    return sd[0] > 0 and sd[0] >= 2 * sd[1:].max()
fb = pd.DataFrame([{"cell": p["cell"], "file": p["file"].split("/")[-1], "probe": p["probe"], "contexts": p["contexts"], "rep": p["rep"],
                    "sd_bin0": p["sd"][0], "sd_max_later": p["sd"][1:-1].max(), "first_bin_only": first_bin(p)} for p in MGB])
design = fb[fb.contexts.map(lambda c: len(c) == 3 and asari.BLANK in c)]
print(f"MGB probes with silence and two fragments as contexts: {len(design)}; of these with the first bin at least twice every later bin: {int(design.first_bin_only.sum())}")
print(design.round(2).to_string(index=False))
print(f"\nAll MGB probes with the pattern: {int(fb.first_bin_only.sum())} of {len(fb)} ({fb.first_bin_only.mean():.2f})")
for factor in (1.5, 2, 3):
    k = sum(p["sd"][0] > 0 and p["sd"][0] >= factor * p["sd"][1:-1].max() for p in MGB)
    kd = sum(p["sd"][0] > 0 and p["sd"][0] >= factor * p["sd"][1:-1].max() for p in MGB if len(p["contexts"]) == 3 and asari.BLANK in p["contexts"])
    print(f"Factor {factor}: {kd} of {len(design)} probes of the Figure 9 design, {k} of {len(fb)} MGB probes")

**Result: reproduced.** Of the 12 MGB probes tested with silence and two fragments, the 4 of cell 20071010-ha2-003 (10 trials) have an SD over contexts in the first bin at least twice that of every later bin; over all MGB probes 36 of 93 (0.39) show this. The factor of two is the claims file's reading: with a factor of 1.5 the same 4 probes qualify, with a factor of 3 none of the 12 does.

## figs1-silence-probe

> A subset of cells showed significant context dependence soon after the onset of the
> intersequence silences, but the response power attributable to the preceding stimulus alone
> faded within about 1 s after the stimulus ended.

Silence probes are the first 2.5 s after each sequence, with the sequence's last fragment as the
context (`CxtDep.m`). The significance measure; and the population average of Eq. 17 (main) and
Eq. 19 (variant) over the silence probes, against its value in the first 10 ms.

In [ ]:
cells_soon = sorted({p["cell"] for p in sil if any(a < 0.5 for a, b in asari.runs(p["signif"]))})
print(f"Silence probes: {len(sil)} in {len({p['cell'] for p in sil})} cells; significant at all: {sum(bool(p['signif'].any()) for p in sil)}; "
      f"cells with a significant period starting within 0.5 s of silence onset: {len(cells_soon)}")
Ls = min(p["T"] for p in sil)
pred_s = np.mean([p["pred"][:Ls] for p in sil], axis=0)
ctx_s = np.mean([(p["pred"] - p["inv"])[:Ls] for p in sil], axis=0)
for name, c in (("Eq. 17, stimulus-related", pred_s), ("Eq. 19, context-dependent", ctx_s)):
    b = asari.bin_mean(c, 100)
    onset = b[0]
    below = np.where(b < 0.1 * onset)[0]
    first = (below[0] + 1) * 0.01 if len(below) else np.nan
    stays = bool(np.all(b[110:] < 0.1 * onset))
    print(f"{name}: first 10 ms {onset:.2f} mV^2; first 10-ms bin below 10%: ends at {first:.2f} s; mean 1.0-1.1 s {b[100:110].mean():.2f}; "
          f"every bin after 1.1 s below 10%: {stays}; mean over 2-2.5 s {b[200:].mean():.2f}")

**Result: reproduced.** 19 of the 38 cells with a silence probe have a significant period starting within 0.5 s of the silence onset. The stimulus-related power over the 44 silence probes is 5.60 mV^2 in the first 10 ms and falls below 10% of that by 0.55 s; it averages -0.09 mV^2 from 1.0 to 1.1 s, though some later bins rise above 10% again (mean 0.45 mV^2 from 2 to 2.5 s).

## fig7-example-frequency-context

> In an example A1 neuron, a natural sound probe preceded by three DMR contexts that differed only
> in frequency band gave significant context dependence for about 4 s.

Search for a recording whose conditioning stimuli are DMRs in 0.625-2.5, 2.5-10 and 10-40 kHz with a
natural sound probe. The names, descriptions, bands (`ff`), SoundTrans shifts (`df`) and
attenuations of every DMR, TORC and SoundTrans recording are printed. Each DMR also gets a CRC-32
checksum of its modulation parameters (`tempmod`, `specmod`, `SFtemp`, `SFspec`, `envphase`, `sinphase`,
`dx`), so that the DMRs that share them can be seen: the frequency variants of a recording are one
DMR in different bands only if they share the checksum. As a weaker check that does not need the example, the natural sound probes
tested with DMR frequency variants as the only contexts are listed with the end of their last
significant period.

In [ ]:
print("Stimuli of every DMR, TORC and SoundTrans recording in group 2 (bands 'ff' in Hz; DMRipple2.m reads a 4-element ff as two bands; "
      "'df' is the SoundTrans frequency shift; 'modulation' a CRC-32 of a DMR's modulation parameters " + ", ".join(asari.DMR_MOD_FIELDS) + "), "
      "grouped by identical stimulus lists:")
groups7 = {}
for r in results:
    if r.get("stimuli") is None or not set(r["info"]["trigger_types"]) & {"dmr", "torc", "naturalistic"}:
        continue
    tabs = pd.DataFrame(r["stimuli"])
    cols = [c for c in ["stim", "name", "description", "ff", "df", "attenuation", "modulation"] if c in tabs]
    groups7.setdefault(tabs[cols].astype(str).to_csv(index=False), []).append((r, tabs[cols]))
target = {(625.0, 2500.0), (2500.0, 10000.0), (10000.0, 40000.0)}
n_match = 0
for key, rs in groups7.items():
    r0, tabs = rs[0]
    print(f"\n=== {r0['info']['trigger_types']} {len(rs)} recordings: " + ", ".join(x["file"].split("/")[-1] for x, _ in rs))
    print(tabs.to_string(index=False))
    singles = {tuple(ff) for ff in tabs.get("ff", pd.Series(dtype=object)).dropna() if len(ff) == 2}
    if target <= singles:
        n_match += len(rs)
print("\nDMR frequency variants: per recording, the distinct bands and the distinct modulation checksums among them:")
for r in results:
    for st in [pd.DataFrame(r["stimuli"])] if r.get("stimuli") is not None and "dmr" in r["info"]["trigger_types"] else []:
        fv = st[st.description.str.contains("frequency variants")]
        if len(fv):
            print(f"  {r['file'].split('/')[-1]}: {len(fv)} stimuli, {fv.ff.nunique()} bands, {fv.modulation.nunique()} modulation checksum(s) {sorted(fv.modulation.unique())}")
print(f"\nRecordings with three conditioning stimuli in 0.625-2.5, 2.5-10 and 10-40 kHz, each a single band: {n_match}")

print("\nWeaker check: natural sound probes tested with DMR frequency variants as the only contexts (every set), and the end of their last significant period:")
w7 = pd.DataFrame([{"file": s["file"].split("/")[-1], "probe": s["probe"], "set": s["set"], "contexts": s["contexts"], "trials": s["rep"], "T_s": s["T"] / SF,
                    "last_significant_end_s": max([b for a, b in asari.runs(s["signif"])], default=np.nan)}
                   for s in SY_ALL if s["property"] == "frequency" and s["ensemble"] == "dmr" and s["probe_type"] == "natural"])
print(w7.round(2).to_string(index=False))
print(f"Tests with a significant period ending at least 3.6 s after probe onset: {int((w7.last_significant_end_s >= 3.6).sum())} of {len(w7)} "
      f"(in {w7[w7.last_significant_end_s >= 3.6].file.nunique()} recordings)")

**Result: not testable here.** No recording in the data set has DMR conditioning stimuli in 0.625-2.5, 2.5-10 and 10-40 kHz, each a single band. The DMR frequency variants in it are one DMR (one modulation checksum in each of the 5 recordings) in six 2-octave bands from 0.25 to 32 kHz in 1-octave steps (5 recordings), or the broad and two-band stimuli 625-10000, 1250-20000, 2500-40000 and 625-2500 with 10000-40000 Hz (5 recordings). In the weaker check, 3 of 10 natural sound probe tests with DMR frequency variants as contexts (in 3 recordings) have a significant period ending at least 3.6 s after probe onset.

## Summary

| Claim | Outcome |
|---|---|
| methods-recording-counts | partially reproduced |
| a1-mgb-firing-rates | partially reproduced |
| fig2-example-reliability | reproduced |
| fig2-example-long-context | reproduced (a selected candidate, not the paper's example cell) |
| fig3a-proportion-significant | reproduced |
| fig3a-significance-decay | partially reproduced |
| fig3b-fractional-power-decay | not reproduced |
| timescale-longer-than-membrane | reproduced |
| fig4-predictability-upper-bound | not reproduced |
| fig4-strf-performance-low | not reproduced (assumed cell selection and test; `strf_wehr.ipynb`) |
| fig5-example-conductances | blocked: data missing |
| fig6-conductance-decay | blocked: data missing |
| fig7-example-frequency-context | blocked: data missing |
| fig8a-proportion-by-property | partially reproduced (assumed property sets) |
| fig8a-significance-fits | not reproduced (assumed property sets) |
| fig8b-decay-by-property | partially reproduced (assumed property sets) |
| fig8c-relative-effects | not reproduced (assumed property sets) |
| fig8-natural-synthetic-probes-alike | reproduced (assumed test and property sets) |
| fig9-mgb-fast-decay | reproduced |
| fig9-example-first-bin | reproduced |
| figs1-silence-probe | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The probe responses, the significance measure and
the fractional powers follow the paper's own script `CxtDep.m`, which is in the data set. The
population curves average probes of different lengths, which the paper leaves open; both readings
are shown. The A1 spike detector is a choice (the paper's code detects A1 spikes by hand). The
property sets of Figure 8 are rebuilt from each recording's stimulus metadata, with one test per
set for AM and FM, the reading that gives the probe totals of Table 1. The trial
resampling reading of Figure 8C is run in 20-ms bins; its resampled areas are biased upwards. The STRF and LN models of Figure 4 are fitted in
`strf_wehr.ipynb` to the 16 of Wehr's 25 cells with repeated stimuli (the paper used 20, not listed).

**What was found about the data set.** It holds the paper's 39 natural sound A1 cells (305
probes) and 14 MGB cells (93 probes) exactly, 96 synthetic sound A1 cells, and 12 unpublished
cells, with the paper's analysis code. It holds no voltage-clamp recording, so Figures 5 and 6
cannot be tested. Two cells (20060314-ha1-001 and 20060320-ha3-003) are listed in both the natural and the synthetic sound group of `Summary.mat`, so their tuning-curve files appear twice; each is counted once here. Which
conditioning stimuli form each property set of Figure 8 is documented in the notes of only the
19-stimulus MHT recordings; elsewhere it has to be rebuilt from the stimulus parameters. Only 2
recordings have MHT probes; the other MHT recordings test a natural sound probe.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")